# EEG Representation Learning for Brain Signal Decoding (ZuCo 1.0 & ZuCo 2.0)
**End-to-End Research Pipeline: From Raw EEG to Language Decoding & Representation Robustness**

This Colab notebook is designed for reproducible execution with:
1. Google Drive mounting and automated checkpoint/results persistence.
2. Step-by-step modular pipeline with resumability.
3. Smoke test mode (quick verification with 2-3 subjects) vs full-scale experiments.
4. Memory-mapped / streaming data loading to prevent RAM crashes.

## 0. Hardware & GPU Diagnostics

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 1. Mount Google Drive & Configure Paths
Set `DRIVE_DATA_DIR` to the folder in your Google Drive where your preprocessed ZuCo files reside.

In [ ]:
from google.colab import drive
import os
from pathlib import Path

# Mount Drive
drive.mount('/content/drive')

# =====================================================================
# USER CONFIGURATION: Set your Drive paths here
# =====================================================================
DRIVE_DATA_DIR = "/content/drive/MyDrive/ZuCo"  # <--- Change if different
DRIVE_EXP_DIR  = "/content/drive/MyDrive/eeg_language_decoding"

LOCAL_PROJECT_DIR = "/content/eeg_language_decoding"

print(f"Checking Drive data directory: {DRIVE_DATA_DIR}")
if os.path.exists(DRIVE_DATA_DIR):
    print("SUCCESS: Drive data directory found!")
    print("Contents preview:", os.listdir(DRIVE_DATA_DIR)[:10])
else:
    print(f"WARNING: Drive data path not found. Please verify the folder in your Google Drive.")

# Create experiment output directory in Drive for persistent checkpoints and figures
os.makedirs(DRIVE_EXP_DIR, exist_ok=True)
for sub in ["checkpoints", "results", "figures", "docs"]:
    os.makedirs(os.path.join(DRIVE_EXP_DIR, sub), exist_ok=True)

## 2. Environment Setup & Dependency Installation

In [ ]:
# Install project dependencies
!pip install -q mne sentence-transformers transformers h5py pyyaml scikit-learn seaborn

# Navigate to project directory
import sys
if str(LOCAL_PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(LOCAL_PROJECT_DIR))
os.environ["PYTHONPATH"] = f"{LOCAL_PROJECT_DIR}:{os.environ.get('PYTHONPATH', '')}"

## 3. STEP 0: Mandatory Data Inspection
This script recursively scans your ZuCo files (both `.mat` v5/v7/v7.3 and `.pkl` formats), extracts real subject IDs, tasks, sampling rates, channel counts, sentence durations, and word-level structures, and saves:
1. `docs/data_inspection.md` - Complete markdown report with ZuCo 1 vs ZuCo 2 reconciliation.
2. `data/zuco_unified_index.csv` - Master index linking all trials.

Everything is automatically copied to your Google Drive (`DRIVE_EXP_DIR`) so it is never lost.

In [ ]:
# Execute Step 0: Data Inspection
!python src/scripts/step0_inspect_data.py \
    --data_dir "$DRIVE_DATA_DIR" \
    --output_doc "docs/data_inspection.md" \
    --output_csv "data/zuco_unified_index.csv"

# Copy inspection outputs directly to Drive for persistence
!cp docs/data_inspection.md "$DRIVE_EXP_DIR/docs/data_inspection.md"
!cp data/zuco_unified_index.csv "$DRIVE_EXP_DIR/data/zuco_unified_index.csv"
print("Copied inspection report and index to Google Drive!")

### Display the Generated Inspection Report & Unified Index Preview

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

# Display markdown report
if os.path.exists("docs/data_inspection.md"):
    with open("docs/data_inspection.md", "r", encoding="utf-8") as f:
        display(Markdown(f.read()))

# Preview unified index table
if os.path.exists("data/zuco_unified_index.csv"):
    df_idx = pd.read_csv("data/zuco_unified_index.csv")
    print(f"\nTotal index entries: {len(df_idx)}")
    display(df_idx.head(10))

## Next Steps
Once Step 0 completes in Colab, **copy the terminal summary output and paste it back into your chat with Antigravity**.
We will immediately verify the real file structure and initiate Step 1 (Data preparation & deterministic splits) and Step 2 (Standardized EEG preprocessing).